# 🚀 SpaceX Launch Analysis — Lesson 2: Visualizations

**Goal:** Turn the numbers from Lesson 1 into charts that tell a story.

---

## The libraries

- **matplotlib** — the base charting library in Python. Powerful but verbose.
- **seaborn** — built on top of matplotlib. Less code, better-looking defaults.

Think of it like this: matplotlib is the engine, seaborn is the nicer car built on top of it. We'll use both.

We'll build 4 charts:
1. Launch frequency over time (bar chart)
2. Success rate by rocket type (horizontal bar)
3. Payload mass distribution (histogram)
4. Booster reuse trend (line chart)

---

## Setup — load everything we need

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# Set a clean visual style globally — applies to all charts
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)  # default chart size

# Load the data (same as Lesson 1)
df = pd.read_csv('../data/spacex_launches.csv')
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year

print(f'Data loaded: {df.shape[0]} launches')
print('✅ Libraries ready')

---

## Chart 1: Launch Frequency Over Time

**Question:** How has SpaceX's launch cadence grown year over year?

**Chart type:** Bar chart — good for comparing discrete categories (years).

**Pattern:**
1. Prepare the data (aggregate with pandas)
2. Create the figure
3. Add labels and formatting
4. Show it

You'll follow this same 4-step pattern for every chart you ever make.

In [ ]:
# Step 1: Prepare — count launches per year
launches_per_year = df.groupby('year').size().reset_index(name='count')
# groupby('year') — group rows by year (like SQL GROUP BY)
# .size()         — count rows in each group
# .reset_index()  — turn the result back into a normal DataFrame

print(launches_per_year)

In [ ]:
# Steps 2-4: Build the chart
fig, ax = plt.subplots()
# fig = the whole figure (the canvas)
# ax  = the axes object (where the actual chart lives)
# This pattern lets you control every detail of the chart

bars = ax.bar(
    launches_per_year['year'],   # x-axis: years
    launches_per_year['count'],  # y-axis: launch count
    color='steelblue',
    edgecolor='white',
    linewidth=0.5
)

# Add the count number on top of each bar
ax.bar_label(bars, padding=3, fontsize=9)

# Labels and title
ax.set_title('SpaceX Launches Per Year (2006–2024)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Year')
ax.set_ylabel('Number of Launches')
ax.set_xticks(launches_per_year['year'])
ax.set_xticklabels(launches_per_year['year'], rotation=45)

plt.tight_layout()  # prevents labels from getting cut off
plt.show()

**🧠 What does this tell us?**
SpaceX went from 1 launch in 2006 to a massive ramp-up post-2020. That spike is almost entirely driven by Starlink — they needed to launch hundreds of satellites to build out the constellation. Launch cadence is a direct proxy for company maturity.

---

## Chart 2: Success Rate by Rocket Type

**Question:** Which rockets had the best track record?

**Chart type:** Horizontal bar — better than vertical when labels are long.

In [ ]:
# Prepare: calculate success rate per rocket
rocket_stats = df.groupby('rocket').agg(
    total=('success', 'count'),      # count all launches
    successes=('success', 'sum')     # sum True values (True=1)
).reset_index()

rocket_stats['success_rate'] = (rocket_stats['successes'] / rocket_stats['total'] * 100).round(1)

# Sort by success rate descending
rocket_stats = rocket_stats.sort_values('success_rate', ascending=True)  # ascending=True for horizontal bars

print(rocket_stats)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Color bars based on success rate
colors = ['#e74c3c' if r < 70 else '#f39c12' if r < 90 else '#2ecc71' 
          for r in rocket_stats['success_rate']]
# List comprehension: for each rate, pick red/orange/green

bars = ax.barh(
    rocket_stats['rocket'],        # y-axis: rocket names
    rocket_stats['success_rate'],  # x-axis: success rate
    color=colors,
    edgecolor='white'
)

# Add percentage labels at end of each bar
for bar, rate, total in zip(bars, rocket_stats['success_rate'], rocket_stats['total']):
    ax.text(
        bar.get_width() + 0.5,          # x position (just past end of bar)
        bar.get_y() + bar.get_height()/2,  # y position (middle of bar)
        f'{rate}%  ({total} launches)',
        va='center', fontsize=10
    )

ax.set_xlim(0, 115)  # give space for labels
ax.set_title('Mission Success Rate by Rocket Type', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Success Rate (%)')
ax.set_ylabel('')
ax.axvline(x=90, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='90% threshold')
ax.legend()

plt.tight_layout()
plt.show()

**🧠 What does this tell us?**
Falcon 1 had a rough start (only 2/5 succeeded) — expected for a brand new rocket. Falcon 9 Block 5 became their gold standard. Starship is still in testing so its "success" is partial. The improvement from Falcon 1 → Falcon 9 → Block 5 shows the learning curve of rocket development.

---

## Chart 3: Payload Mass Distribution

**Question:** What does the typical payload weight look like? Is it evenly spread or clustered?

**Chart type:** Histogram — shows the distribution of a continuous variable.

This is where seaborn shines — one line of code.

In [ ]:
# Filter out 0 kg payloads (classified missions with unknown mass)
payload_data = df[df['payload_kg'] > 0]

fig, ax = plt.subplots()

sns.histplot(
    data=payload_data,
    x='payload_kg',
    bins=20,           # number of buckets
    kde=True,          # adds a smooth curve on top (kernel density estimate)
    color='steelblue',
    ax=ax
)

# Add a vertical line at the median
median_payload = payload_data['payload_kg'].median()
ax.axvline(median_payload, color='red', linestyle='--', linewidth=1.5, label=f'Median: {median_payload:,.0f} kg')

ax.set_title('Payload Mass Distribution', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Payload Mass (kg)')
ax.set_ylabel('Number of Launches')
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend()

plt.tight_layout()
plt.show()

**🧠 What does this tell us?**
There are two clusters — lighter payloads (individual satellites, ISS resupply) and a big spike near 18,500 kg. That spike is Starlink batches. The distribution isn't random noise — it reflects specific mission types. This is what "reading" a histogram looks like.

---

## Chart 4: Booster Reuse Trend

**Question:** How quickly did SpaceX ramp up booster reuse?

**Chart type:** Line chart — shows change over time (trends).

In [ ]:
# Calculate % of launches using reused boosters per year
reuse_trend = df.groupby('year')['reused'].mean().mul(100).reset_index()
reuse_trend.columns = ['year', 'reuse_pct']

fig, ax = plt.subplots()

ax.plot(
    reuse_trend['year'],
    reuse_trend['reuse_pct'],
    marker='o',          # dot at each data point
    linewidth=2.5,
    markersize=8,
    color='#e67e22',
    markerfacecolor='white',
    markeredgewidth=2
)

# Shade the area under the line
ax.fill_between(reuse_trend['year'], reuse_trend['reuse_pct'], alpha=0.15, color='#e67e22')

# Annotate the first reuse milestone
ax.annotate(
    'First reused\nbooster flight',
    xy=(2017, reuse_trend[reuse_trend['year']==2017]['reuse_pct'].values[0]),
    xytext=(2015, 60),
    arrowprops=dict(arrowstyle='->', color='gray'),
    fontsize=9,
    color='gray'
)

ax.set_ylim(0, 110)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f'{y:.0f}%'))
ax.set_xticks(reuse_trend['year'])
ax.set_xticklabels(reuse_trend['year'], rotation=45)
ax.set_title('Booster Reuse Rate Over Time', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Year')
ax.set_ylabel('% of Launches Using Reused Booster')

plt.tight_layout()
plt.show()

**🧠 What does this tell us?**
Reuse went from 0% to nearly 100% in just a few years. This is arguably SpaceX's most important engineering achievement — reusing boosters is what makes the economics of space access work. Each reused booster saves an estimated $30–50M. By 2022 they were reusing boosters on virtually every flight.

---

## Bonus: All 4 charts in one figure

This is what a real portfolio dashboard looks like — multiple charts in a single layout using `plt.subplots()`.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('SpaceX Launch Analysis (2006–2024)', fontsize=18, fontweight='bold', y=1.01)

# --- Chart 1: Launches per year ---
axes[0,0].bar(launches_per_year['year'], launches_per_year['count'], color='steelblue', edgecolor='white')
axes[0,0].set_title('Launches Per Year')
axes[0,0].set_xlabel('Year')
axes[0,0].set_ylabel('Count')
axes[0,0].tick_params(axis='x', rotation=45)

# --- Chart 2: Success rate by rocket ---
axes[0,1].barh(rocket_stats['rocket'], rocket_stats['success_rate'], color=colors, edgecolor='white')
axes[0,1].set_title('Success Rate by Rocket')
axes[0,1].set_xlabel('Success Rate (%)')
axes[0,1].axvline(x=90, color='gray', linestyle='--', linewidth=1, alpha=0.5)

# --- Chart 3: Payload distribution ---
sns.histplot(data=payload_data, x='payload_kg', bins=20, kde=True, color='steelblue', ax=axes[1,0])
axes[1,0].axvline(median_payload, color='red', linestyle='--', linewidth=1.5)
axes[1,0].set_title('Payload Mass Distribution')
axes[1,0].set_xlabel('Payload (kg)')

# --- Chart 4: Reuse trend ---
axes[1,1].plot(reuse_trend['year'], reuse_trend['reuse_pct'], marker='o', linewidth=2.5, color='#e67e22')
axes[1,1].fill_between(reuse_trend['year'], reuse_trend['reuse_pct'], alpha=0.15, color='#e67e22')
axes[1,1].set_title('Booster Reuse Rate Over Time')
axes[1,1].set_xlabel('Year')
axes[1,1].set_ylabel('% Reused')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../outputs/spacex_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

print('✅ Dashboard saved to outputs/spacex_dashboard.png')

---

## 🎯 Lesson 2 Summary

| Concept | What you learned |
|---|---|
| `fig, ax = plt.subplots()` | The standard way to create a chart |
| `ax.bar()` / `ax.barh()` | Vertical and horizontal bar charts |
| `ax.plot()` | Line chart |
| `sns.histplot()` | Histogram with optional KDE curve |
| `ax.set_title/xlabel/ylabel` | Labeling charts |
| `ax.axvline()` / `ax.axhline()` | Reference lines |
| `plt.subplots(2,2)` | Multi-chart dashboard layout |
| `plt.savefig()` | Export chart to file |
| List comprehension for colors | Conditional formatting on bars |

**The 4-step pattern for every chart:**
1. Prepare the data (aggregate/filter with pandas)
2. Create the figure (`fig, ax = plt.subplots()`)
3. Add labels and formatting
4. Show or save it

---

## ➡️ What's next: Lesson 3 — Data Cleaning & Feature Engineering

We'll handle those 24 missing `landing_success` values, create new columns from existing data (like decade, era, payload category), and prep the dataset for the final analysis.